# HPC Job Schedulers

On shared HPC clusters, users cannot directly access compute nodes interactively. A **job scheduler** acts as a resource controller: it accepts job requests, manages queues, enforces fair allocation across users, and assigns work to available nodes when resources become free.

This notebook introduces the two schedulers most relevant to executorlib users:
- **SLURM** — the dominant scheduler on most HPC systems today
- **Flux** — a modern hierarchical scheduler that can run inside a SLURM allocation

It then shows how executorlib's different submission modes map onto these schedulers.

## SLURM

The [Simple Linux Utility for Resource Management (SLURM)](https://slurm.schedmd.com) is the job scheduler on most large HPC systems, including LANL's Chicoma and Venado. Three commands cover the day-to-day workflow:

### `sbatch` — submit a batch job

`sbatch` submits a shell script from the login node to the scheduler. The script contains resource directives (lines starting with `#SBATCH`) and the commands to run.

```bash
#!/bin/bash
#SBATCH --job-name=my_job
#SBATCH --output=my_job.out
#SBATCH --ntasks=4           # number of MPI tasks
#SBATCH --cpus-per-task=1    # CPU threads per task
#SBATCH --time=00:30:00      # wall-clock limit (HH:MM:SS)
#SBATCH --partition=regular

srun python my_script.py
```

```bash
sbatch job.sh          # submit — returns a job ID immediately
```

Key `#SBATCH` directives:

| Directive | Meaning |
|---|---|
| `--ntasks=N` | Total MPI ranks (processes) |
| `--cpus-per-task=N` | CPU threads available to each rank |
| `--mem=NG` | Memory per node (e.g. `8G`) |
| `--time=HH:MM:SS` | Maximum wall-clock time |
| `--partition=name` | Queue / partition to target |
| `--dependency=afterok:JOBID` | Run only after another job succeeds |

### `srun` — launch parallel tasks inside an allocation

`srun` starts one or more parallel tasks *within* an existing allocation (e.g. inside a batch script or an interactive session). It is the command executorlib uses internally for MPI-parallel Python functions.

```bash
# run 4 MPI ranks of a Python script using the PMIx launcher
srun --mpi=pmix -n 4 python my_mpi_script.py
```

Multiple `srun` calls inside one batch script can run sequentially or in parallel (using shell backgrounding with `&` and `wait`):

```bash
srun -n 4 python task_a.py &
srun -n 4 python task_b.py &
wait   # block until both finish
```

### `squeue` and `sacct` — inspect jobs

`squeue` shows jobs currently in the scheduler queue:

```bash
squeue --me            # your jobs only
squeue -j 12345        # one specific job
```

Common state codes: `PD` (pending), `R` (running), `CG` (completing).

`sacct` queries the accounting database for completed or running jobs — useful for verifying which resources were actually allocated:

```bash
sacct -j 12345                          # summary for job 12345
sacct -j 12345 --format=JobID,State,AllocCPUS,Elapsed
```

### MPI-parallel Python with `mpi4py`

The [Message Passing Interface (MPI)](https://www.mpi-forum.org) is the dominant parallelisation standard on HPC systems. [`mpi4py`](https://mpi4py.readthedocs.io) provides Python bindings. A minimal example:

```python
# script.py
from mpi4py import MPI

comm = MPI.COMM_WORLD
print(f"rank {comm.Get_rank()} of {comm.Get_size()}")
```

Launch with:

```bash
srun --mpi=pmix -n 4 python script.py
```

When multiple independent groups of ranks need to run inside one allocation there are two approaches:

| Approach | How | Cross-group communication |
|---|---|---|
| Multiple `srun` calls | Each `srun` launches its own communicator | Not possible |
| `MPI_Comm_split` | One `srun`, communicator split in Python | Possible via `MPI.COMM_WORLD` |

```python
# communicator splitting — 8 ranks split into two groups of 4
comm = MPI.COMM_WORLD
color = comm.Get_rank() // 4   # group 0 or group 1
sub_comm = comm.Split(color)
```

## Flux

[Flux](http://flux-framework.org) is a modern HPC resource manager developed at Lawrence Livermore National Laboratory (LLNL). On many LANL systems it runs as a **secondary scheduler within a SLURM allocation**, enabling fine-grained hierarchical task distribution without submitting individual `sbatch` jobs.

Flux can also be installed locally via conda for development and testing — unlike SLURM, it requires no system-level configuration:

```bash
conda install -c conda-forge flux-core
flux start   # launch a local Flux instance
```

### Key Flux commands

| Command | SLURM equivalent | Description |
|---|---|---|
| `flux resource list` | `sinfo` | Show available nodes, cores, GPUs |
| `flux jobs -a` | `squeue` | List all jobs (running and completed) |
| `flux submit` | `sbatch` | Submit a non-blocking job; returns a job ID immediately |
| `flux run` | `srun` | Launch a blocking job; waits for completion |
| `flux job attach <ID>` | — | Stream output of a previously submitted job |

```bash
# inspect available resources
flux resource list

# submit a non-blocking 4-rank MPI job
flux submit -o pmi=pmix --ntasks=4 python script.py

# run a blocking 4-rank MPI job (waits for output)
flux run -o pmi=pmix -n 4 python script.py

# stream output of a previously submitted job
flux job attach <jobid>
```

The `-o pmi=pmix` flag tells Flux to use the PMIx process-management interface, matching what SLURM's `--mpi=pmix` provides — the same `mpi4py` scripts run unchanged under both schedulers.

## executorlib Submission Modes

executorlib exposes three executor classes that map onto different levels of the scheduler hierarchy. The choice depends on whether you are submitting from a login node or from within an already-running allocation.

| Class | Scheduler command | Typical use |
|---|---|---|
| `SlurmClusterExecutor` | `sbatch` (one job per function call) | Submit from login node; each Python function becomes an independent SLURM job |
| `SlurmJobExecutor` | `srun` steps within the current job | Run inside an existing SLURM allocation; no new jobs are queued |
| `FluxClusterExecutor` | `flux submit` | Submit from login node or within a Flux instance; good for local testing |

### `SlurmClusterExecutor` — one sbatch job per function call

`SlurmClusterExecutor` is the right choice when submitting workflows from the **login node**. Each call to `exe.submit()` results in one `sbatch` submission. Results are serialised to a cache directory on disk so the Python process can exit and the results reloaded later.

```python
from executorlib import SlurmClusterExecutor

with SlurmClusterExecutor(cache_directory="./cache") as exe:
    future_lst = [exe.submit(sum, [i, i]) for i in range(1, 4)]
    print([f.result() for f in future_lst])
```

Resource parameters — such as maximum run time, memory, partition, and the submission template itself — can be passed per function via `resource_dict`:

```python
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env=L
#SBATCH --partition={{partition}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency %}
#SBATCH --dependency=afterok:{{ dependency | join(',') }}
{%- endif %}
{%- if memory_max %}
#SBATCH --mem={{memory_max}}G
{%- endif %}
#SBATCH --ntasks={{cores}}

{{command}}
"""

def get_env():
    import os
    return {k: v for k, v in os.environ.items() if "SLURM_" in k}

with SlurmClusterExecutor(cache_directory="./cache") as exe:
    future = exe.submit(
        get_env,
        resource_dict={
            "submission_template": submission_template,
            "run_time_max": 180,   # seconds
            "partition": "regular",
            "cores": 1,
        })
    print(future.result())
```

The template uses [Jinja2](https://jinja.palletsprojects.com) syntax. executorlib fills `{{cores}}` into `--ntasks`, so `cores=1` requests one serial process and `cores=4` requests four MPI ranks.

### MPI-parallel functions with `SlurmClusterExecutor`

To submit an MPI-parallel Python function, set `cores` to the number of ranks and pass `pmi_mode="pmix"` to the executor. executorlib then wraps the function in `srun --mpi=pmix -n <cores>`.

```python
def mpi_calc(i):
    from mpi4py import MPI
    comm = MPI.COMM_WORLD
    return {"rank": comm.Get_rank(), "size": comm.Get_size(), "input": i}

with SlurmClusterExecutor(pmi_mode="pmix", cache_directory="./cache") as exe:
    future = exe.submit(
        mpi_calc, 42,
        resource_dict={"cores": 4, "partition": "regular", "run_time_max": 120})
    print(future.result())
```

### `SlurmJobExecutor` — srun steps inside the current allocation

`SlurmJobExecutor` is used **inside a running SLURM job** (e.g. within a batch script or a nested function). Instead of calling `sbatch`, it dispatches each submitted function as an `srun` step within the existing allocation — no new jobs are queued and no waiting for scheduling.

```python
from executorlib import SlurmJobExecutor

# This code runs inside an already-running SLURM job
with SlurmJobExecutor(max_workers=4) as exe:
    futures = [exe.submit(sum, [i, i], resource_dict={"cores": 1}) for i in range(4)]
    print([f.result() for f in futures])
```

### Nested executors

The two executor types compose naturally. A function submitted via `SlurmClusterExecutor` (i.e. running as an `sbatch` job) can itself create a `SlurmJobExecutor` to parallelise sub-tasks using `srun` steps — all within the same allocation:

```python
from executorlib import SlurmClusterExecutor, SlurmJobExecutor

def parallel_workflow(n):
    """Runs as an sbatch job; spawns n srun steps internally."""
    with SlurmJobExecutor(max_workers=n) as inner:
        futures = [inner.submit(sum, [i, i], resource_dict={"cores": 1}) for i in range(n)]
        return [f.result() for f in futures]

with SlurmClusterExecutor(cache_directory="./cache") as outer:
    future = outer.submit(
        parallel_workflow, 4,
        resource_dict={
            "cores": 1,           # outer allocation: 1 process
            "partition": "regular",
            "run_time_max": 300,
        })
    print(future.result())
```

The `sacct` output for such a job will show the outer `sbatch` job together with numbered `srun` steps (e.g. `12345.0`, `12345.1`, …), all completing within the single allocation.

### `FluxClusterExecutor` — Flux-based submission

`FluxClusterExecutor` uses the Flux Python API (`flux.job.submit`) instead of `sbatch`. The interface is identical to `SlurmClusterExecutor`, which makes it easy to switch schedulers or develop locally with Flux before running on a SLURM cluster:

```python
from executorlib import FluxClusterExecutor

def add(a, b):
    return a + b

with FluxClusterExecutor(cache_directory="./cache") as exe:
    futures = [exe.submit(add, i, i) for i in range(4)]
    print([f.result() for f in futures])
```

MPI functions work the same way — set `cores` in the `resource_dict`:

```python
with FluxClusterExecutor(cache_directory="./cache") as exe:
    future = exe.submit(mpi_calc, 3, resource_dict={"cores": 2})
    print(future.result())
```

### Summary

| Question | Answer |
|---|---|
| Submitting from a login node with SLURM? | `SlurmClusterExecutor` |
| Already inside a SLURM job, want parallel sub-tasks? | `SlurmJobExecutor` |
| Using Flux (or testing locally)? | `FluxClusterExecutor` |
| Need MPI inside any executor? | Add `pmi_mode="pmix"` + `"cores": N` in `resource_dict` |
| Need job dependencies? | Pass `Future` objects as arguments — executorlib translates them to `--dependency=afterok:…` |

All three executors share the same `submit()` / `future.result()` interface from Python's standard [`concurrent.futures`](https://docs.python.org/3/library/concurrent.futures.html), so switching between local execution, SLURM, and Flux requires changing only the executor class.